# Chunking Experiment

Comparing 3 chunking strategies on a real document against 3 queries:
1. Fixed size with no overlap
2. Fixed size with overlap
3. Semantic (paragraph or section-based)

Considering:
- Chunk size parameters
- Strategy strengths and weaknesses per prompt
- How does chunking introduce problems (split context, loss of definition, duplication)
- Productionization considerations

## Document

**Source:** ["Attention Is All You Need" (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762) — the original Transformer paper. Domain: NLP / deep learning research.

Chosen because it's directly on-topic for this course and has real structure (numbered sections, dense equations and definitions) that fixed-size chunking is likely to visibly disrupt. Text was extracted from the [ar5iv](https://ar5iv.labs.arxiv.org/html/1706.03762) HTML rendering, with section headings preserved as markdown (`##`/`###`/`####`) and figures/tables/bibliography stripped, then saved locally as `attention_is_all_you_need.txt`.

This notebook is retrieval-only — no generation step — since the assignment is about which chunks get retrieved, not about answer quality.

In [5]:
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
DOC = pathlib.Path('attention_is_all_you_need.txt').read_text(encoding='utf-8')
print('doc chars:', len(DOC))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

doc chars: 26237


## Chunking strategies

Three strategies, same document:
- **Fixed, no overlap** — 500 characters per chunk, cut wherever the character count lands.
- **Fixed, with overlap** — 500 characters per chunk, 100-character overlap between consecutive chunks.
- **Semantic (section-based)** — split on markdown headings (`##`/`###`/`####`), so each chunk is one full paper section or subsection, whatever its length.

In [6]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out

def chunk_semantic(text):
    # split right before each section/subsection heading; each chunk keeps its own heading
    parts = re.split(r'\n(?=#{2,4} )', text)
    chunks = []
    for p in parts:
        p = re.sub(r'\s+', ' ', p).strip()
        if not p or (p.startswith('# ') and not p.startswith('## ')):
            continue  # drop the lone document title fragment
        chunks.append(p)
    return chunks

configs = {
    'fixed-no-overlap (500/0)': chunk_fixed(DOC, 500, 0),
    'fixed-overlap (500/100)': chunk_fixed(DOC, 500, 100),
    'semantic (by section)': chunk_semantic(DOC),
}
for name, chunks in configs.items():
    sizes = [len(c) for c in chunks]
    print(f'{name:28s} -> {len(chunks):3d} chunks, size range {min(sizes)}-{max(sizes)}')

fixed-no-overlap (500/0)     ->  53 chunks, size range 148-500
fixed-overlap (500/100)      ->  66 chunks, size range 148-500
semantic (by section)        ->  23 chunks, size range 10-3413


## Build indexes and retrieve

Same embedding model and index type across all three chunk sets, so the only variable is chunking.

In [7]:
def build(chunks):
    e = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx

def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    scores, ids = idx.search(qe, k)
    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [(chunks[i], float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]

indices = {name: build(chunks) for name, chunks in configs.items()}

## Queries

Three queries picked to stress different chunking behaviors against this specific paper:

1. **"What is scaled dot-product attention?"**
2. **"Why did the authors choose self-attention over recurrent and convolutional layers?"**
3. **"What optimizer and learning rate schedule was used for training?"**

In [8]:
queries = [
    'What is scaled dot-product attention?',
    'Why did the authors choose self-attention over recurrent and convolutional layers?',
    'What optimizer and learning rate schedule was used for training?',
]

for q in queries:
    print('=' * 100)
    print('QUERY:', q)
    for name, chunks in configs.items():
        print(f'\n-- {name} --')
        for rank, (chunk, score) in enumerate(retrieve(indices[name], chunks, q, k=3), start=1):
            print(f'  [{rank}] score={score:.3f} len={len(chunk):4d} :: {chunk[:180]}')

QUERY: What is scaled dot-product attention?

-- fixed-no-overlap (500/0) --
  [1] score=0.685 len= 500 :: using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efficient in pract
  [2] score=0.644 len= 500 ::  the weight assigned to each value is computed by a compatibility function of the query with the corresponding key. #### Scaled Dot-Product Attention We call our particular attenti
  [3] score=0.607 len= 500 :: t products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients . To counteract this effect, we scale the dot products by 1 d k

-- fixed-overlap (500/100) --
  [1] score=0.772 len= 500 ::  can depend only on the known outputs at positions less than i . ### Attention An attention function can be described as mapping a query and a set of key-value pairs to an output, 
  [2] score=0.698 len= 500 :: the two mechanisms